# Model Evaluation and Interpretation
**Development of a predictive analytics model for predicting sought-after professions**

This notebook covers:
- Model evaluation metrics
- Confusion matrix analysis
- Feature importance
- ROC curves
- Learning curves

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Custom modules
from src.preprocessing import VacancyPreprocessor
from src.features import FeatureEngineer
from src.models import ProfessionClassifier
from src.evaluation import ModelEvaluator, plot_class_distribution, plot_learning_curves

from sklearn.model_selection import train_test_split

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported!")

## 1. Load Data and Train Model

In [ ]:
# Load and preprocess
preprocessor = VacancyPreprocessor()
df = preprocessor.load_data('../results.csv')
df_processed = preprocessor.preprocess()

# Feature engineering
fe = FeatureEngineer()
X, y, feature_columns = fe.prepare_for_modeling(
    df_processed, target_column='Job', 
    use_tfidf=True, use_skills=True,
    tfidf_max_features=50
)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Class names
class_names = list(fe.label_encoders['target'].classes_)
print(f"Classes: {class_names}")

In [ ]:
# Train best model (Logistic Regression)
model = ProfessionClassifier('logistic')
model.fit(X_train, y_train)

# Create evaluator
evaluator = ModelEvaluator(
    model.model, X_train, X_test, y_train, y_test,
    feature_names=feature_columns, class_names=class_names
)

## 2. Model Metrics

In [ ]:
# Get metrics
metrics = evaluator.get_metrics()

print("=" * 50)
print("MODEL PERFORMANCE METRICS")
print("=" * 50)
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

## 3. Confusion Matrix

In [ ]:
# Plot confusion matrix
import os
os.makedirs('../reports/figures', exist_ok=True)

evaluator.plot_confusion_matrix(
    figsize=(14, 10),
    save_path='../reports/figures/confusion_matrix.png'
)

## 4. Classification Report

In [ ]:
# Text report
from sklearn.metrics import classification_report

print(classification_report(y_test, evaluator.y_pred_test, target_names=class_names))

In [ ]:
# Visual classification report
evaluator.plot_classification_report(
    save_path='../reports/figures/classification_report.png'
)

## 5. ROC Curves

In [ ]:
# ROC curves
evaluator.plot_roc_curves(
    save_path='../reports/figures/roc_curves.png'
)

## 6. Class Distribution

In [ ]:
# Class distribution in train and test
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Train
unique, counts = np.unique(y_train, return_counts=True)
axes[0].bar(range(len(unique)), counts, color='steelblue')
axes[0].set_xticks(range(len(unique)))
axes[0].set_xticklabels(class_names, rotation=45, ha='right')
axes[0].set_title('Training Set Distribution', fontsize=14)
axes[0].set_ylabel('Count')

# Test
unique, counts = np.unique(y_test, return_counts=True)
axes[1].bar(range(len(unique)), counts, color='coral')
axes[1].set_xticks(range(len(unique)))
axes[1].set_xticklabels(class_names, rotation=45, ha='right')
axes[1].set_title('Test Set Distribution', fontsize=14)
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Learning Curves

In [ ]:
# Learning curves
plot_learning_curves(
    ProfessionClassifier('logistic').model, X_train, y_train, cv=5,
    save_path='../reports/figures/learning_curves.png'
)

## 8. Per-Class Performance Analysis

In [ ]:
# Analyze per-class performance
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    y_test, evaluator.y_pred_test
)

# Create DataFrame
per_class_df = pd.DataFrame({
    'Class': class_names,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
}).sort_values('F1-Score', ascending=False)

print("Per-Class Performance:")
print(per_class_df.to_string(index=False))

In [ ]:
# Visualize per-class metrics
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(class_names))
width = 0.25

bars1 = ax.bar(x - width, precision, width, label='Precision', color='steelblue')
bars2 = ax.bar(x, recall, width, label='Recall', color='coral')
bars3 = ax.bar(x + width, f1, width, label='F1-Score', color='seagreen')

ax.set_xticks(x)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_ylabel('Score')
ax.set_title('Per-Class Performance Metrics', fontsize=14)
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('../reports/figures/per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Error Analysis

In [ ]:
# Find misclassified samples
misclassified_idx = np.where(y_test != evaluator.y_pred_test)[0]

print(f"Total misclassified: {len(misclassified_idx)} out of {len(y_test)}")
print(f"Error rate: {len(misclassified_idx)/len(y_test)*100:.2f}%")

# Analyze misclassifications
if len(misclassified_idx) > 0:
    error_df = pd.DataFrame({
        'True_Class': [class_names[y_test[i]] for i in misclassified_idx],
        'Predicted_Class': [class_names[evaluator.y_pred_test[i]] for i in misclassified_idx]
    })
    
    print("\nMisclassification patterns:")
    print(error_df.value_counts().head(10))

## 10. Summary

In [ ]:
print("=" * 60)
print("MODEL EVALUATION SUMMARY")
print("=" * 60)

print(f"\n📊 Model: Logistic Regression")
print(f"\n📈 Performance:")
print(f"   • Accuracy: {metrics['test_accuracy']:.4f}")
print(f"   • Precision: {metrics['test_precision']:.4f}")
print(f"   • Recall: {metrics['test_recall']:.4f}")
print(f"   • F1-Score: {metrics['test_f1']:.4f}")

print(f"\n📁 Generated Reports:")
print(f"   • reports/figures/confusion_matrix.png")
print(f"   • reports/figures/classification_report.png")
print(f"   • reports/figures/roc_curves.png")
print(f"   • reports/figures/class_distribution.png")
print(f"   • reports/figures/learning_curves.png")
print(f"   • reports/figures/per_class_metrics.png")

print("\n" + "=" * 60)
print("✅ EVALUATION COMPLETE!")
print("=" * 60)